# Data processing

The objective of this notebook is to prepare the dataset for subsequent analysis and modeling.

Based on the findings from the exploratory data analysis (EDA), several preprocessing steps are applied to improve data quality and consistency.

These transformations include handling missing values, correcting data types, removing unnecessary columns, and preparing variables for future analysis.

## Imports

In [3]:
import os
import pandas as pd
from pathlib import Path 

cwd = Path.cwd()

if "notebooks" in cwd.parts:
    project_root = cwd.parents[cwd.parts[::-1].index("notebooks")]
    os.chdir(project_root)

print("Current working directory:", os.getcwd())

Current working directory: c:\Users\Darío\Desktop\OTROS\vector-recommender


## Movie data

In [14]:
MOVIES_PATH = "data/raw/letterboxd_movie_ratings_dataset/movie_data.csv"

ratings_df = pd.read_csv(
    MOVIES_PATH,
    engine="python",
)

ratings_df.head()

,_id,genres,image_url,imdb_id,imdb_link,movie_id,movie_title,original_language,overview,popularity,production_countries,release_date,runtime,spoken_languages,tmdb_id,tmdb_link,vote_average,vote_count,year_released
0,5fc85f606758f69634496fd3,"[""Music"",""Animation""]",film-poster/4/6/4/4/4/0/464440-football-freaks...,NaN,NaN,football-freaks,Football Freaks,en,"Football crazy, football mad. Don’t watch this...",0.600,"[""United Kingdom""]",1971-12-05,0.0,[],535272.0,https://www.themoviedb.org/movie/535272/,0.0,0.0,1971.0
1,5fc85ff26758f696344ace0c,[],film-poster/2/4/5/5/0/0/245500-aftermath-0-230...,tt0586129,http://www.imdb.com/title/tt0586129/maindetails,aftermath-1960,Aftermath,en,Aftermath was the pilot for an unsold TV serie...,0.600,[],1960-04-17,22.0,[],318331.0,https://www.themoviedb.org/movie/318331/,8.0,1.0,1960.0
2,5fc85f606758f69634496fcd,"[""Drama""]",film-poster/9/3/3/1/8/93318-where-chimneys-are...,tt0045731,http://www.imdb.com/title/tt0045731/maindetails,where-chimneys-are-seen,Where Chimneys Are Seen,ja,Gosho’s most celebrated film both in Japan and...,1.568,"[""Japan""]",1953-03-05,108.0,"[""日本語""]",117779.0,https://www.themoviedb.org/movie/117779/,6.6,10.0,1953.0
3,5fc85f606758f69634496fd1,"[""Drama""]",NaN,tt0187327,http://www.imdb.com/title/tt0187327/maindetails,the-musicians-daughter,The Musician's Daughter,en,Carl Wagner's good wife was dying. His heart b...,0.600,"[""United States of America""]",1911-12-12,15.0,[],560377.0,https://www.themoviedb.org/movie/560377/,0.0,0.0,1911.0
4,5fc85f606758f69634496fd4,"[""Documentary""]",film-poster/4/5/4/6/0/3/454603-50-years-of-fab...,tt4769914,http://www.imdb.com/title/tt4769914/maindetails,50-years-of-fabulous,50 Years of Fabulous,en,50 Years of Fabulous recounts the rich history...,0.600,[],2018-05-17,75.0,[],525187.0,https://www.themoviedb.org/movie/525187/,0.0,0.0,2018.0


In [ ]:
PROCESSED_MOVIES_PATH = Path(
    "data/processed/letterboxd_movie_ratings_dataset/movies.csv"
)

PROCESSED_MOVIES_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

ratings_df.to_csv(
    PROCESSED_MOVIES_PATH,
    index=False,
)

print(f"Saved processed ratings dataset to: {PROCESSED_MOVIES_PATH}")

## Ratings Data

In [6]:
RATINGS_PATH = "data/raw/letterboxd_movie_ratings_dataset/ratings_export.csv"

ratings_df = pd.read_csv(
    RATINGS_PATH,
    engine="python",
)

ratings_df.head()

,_id,movie_id,rating_val,user_id
0,5fc57c5d6758f6963451a07f,feast-2014,7,deathproof
1,5fc57c5d6758f6963451a063,loving-2016,7,deathproof
2,5fc57c5d6758f6963451a0ef,scripted-content,7,deathproof
3,5fc57c5d6758f6963451a060,the-future,4,deathproof
4,5fc57c5c6758f69634519398,mank,5,deathproof


Remove films with missing `movie_id`

In [5]:
ratings_df = ratings_df.dropna(subset=["movie_id"]).copy()
ratings_df["movie_id"].isna().sum()
print(f"Ratings after removing missing movie_id: {len(ratings_df):,}")

Ratings after removing missing movie_id: 11,078,161


Save proccesed dataset

In [ ]:
PROCESSED_RATINGS_PATH = Path(
    "data/processed/letterboxd_movie_ratings_dataset/ratings.csv"
)

PROCESSED_RATINGS_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

ratings_df.to_csv(
    PROCESSED_RATINGS_PATH,
    index=False,
)

print(f"Saved processed ratings dataset to: {PROCESSED_RATINGS_PATH}")

Saved processed ratings dataset to: data\processed\letterboxd_movie_ratings_dataset\ratings.csv


## User Data

In [8]:
USERS_PATH = "data/raw/letterboxd_movie_ratings_dataset/users_export.csv"

users_df = pd.read_csv(
    USERS_PATH,
    engine="python",
)

users_df.head()

,_id,display_name,num_ratings_pages,num_reviews,username
0,5fc4172ec6cd28ebd99dd0e2,Lucy,32.0,1650,deathproof
1,5fc4172ec6cd28ebd99dd0ea,Matt Singer,52.0,1915,superpulse
2,5fc4172ec6cd28ebd99dd0ed,Sean Baker,21.0,1283,lilfilm
3,5fc4172ec6cd28ebd99dd0ee,iana,37.0,1177,ianamurray
4,5fc419171ebf67b9fbe48615,Lizzy,57.0,1810,punchdrunklizzy


### Consistency Between Users and Ratings

The ratings dataset contains some users who are not present in the users dataset. To ensure consistency between both tables, we identify usernames that appear in `ratings_df` but are missing from `users_df`.

For each missing user, we add a new entry to `users_df`. Since the original users dataset does not provide information about these users, the unavailable fields are left as missing values. The number of reviews is recovered from the ratings dataset by counting the ratings associated with each missing user.

This allows us to retain all available rating data without discarding users or their interactions.

In [10]:
# Add users present in ratings but missing from the users dataset

rating_usernames = set(ratings_df["user_id"].dropna())
user_usernames = set(users_df["username"].dropna())

missing_usernames = rating_usernames - user_usernames

for username in missing_usernames:
    num_reviews = (
        ratings_df.loc[
            ratings_df["user_id"] == username,
            "rating_val",
        ].count()
    )

    new_user = {
        "_id": pd.NA,
        "display_name": pd.NA,
        "num_ratings_pages": pd.NA,
        "num_reviews": num_reviews,
        "username": username,
    }

    users_df = pd.concat(
        [users_df, pd.DataFrame([new_user])],
        ignore_index=True,
    )

print(
    f"Added {len(missing_usernames)} missing user(s) to users_df."
)

Added 1 missing user(s) to users_df.


Save processed dataset

In [11]:
PROCESSED_RATINGS_PATH = Path(
    "data/processed/letterboxd_movie_ratings_dataset/users.csv"
)

PROCESSED_RATINGS_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

users_df.to_csv(
    PROCESSED_RATINGS_PATH,
    index=False,
)

print(f"Saved processed ratings dataset to: {PROCESSED_RATINGS_PATH}")

Saved processed ratings dataset to: data\processed\letterboxd_movie_ratings_dataset\users.csv
